# Movement timelapse

One full lift (LED on, reach, grasp, lift, hold, replace, return) as an animation: the input EEG on the left and the 3D wrist trajectory on the right, with the saved predictions of several decoders drawn up to a moving cursor.

- Every prediction comes from a saved `predictions.npz` of a LOSO run: the subject shown was the test subject of that fold. Nothing is trained or run here.
- Predictions exist only on the scoring segment [onset - 0.5 s, tHandStop]; outside it only the ground truth is drawn.
- Positions are re-centered on the wrist position at kinematic onset, like the decoder target (cm, lab frame, +z up).
- The ground truth is coloured by phase, so the direction of movement is visible: reach (onset to first touch), lift (first touch to hold start: grasp, load, lift-off, rise), hold (object at or above 90 % of its peak rise above the lift-off height), replace (hold end to release: lower, set down, release), return (release to hand stop); grey before onset and after hand stop. A circle marks the start (wrist at onset), a square the end (wrist at hand stop, shown once reached). The EEG panel shades the same phases. Hold start and end come from the object height, because the dataset has no hold event.

In [1]:
import json
import os
from pathlib import Path

import matplotlib
import pandas as pd
from IPython.display import Markdown, Video, display

if Path.cwd().name == "notebook":
    os.chdir("..")

from eegtraj.data.cache import load_subject
from eegtraj.data.splits import load_split_config, main_subjects
from eegtraj.utils.run import make_run_id
from eegtraj.visualization.timelapse import animate_lift, load_lift, select_lifts

matplotlib.use("Agg")  # videos are rendered off screen and shown below

## Configuration

`EMBED_VIDEO = True` stores each MP4 inside the notebook (base64, not frames), so the videos also play in VS Code. With `False` the notebook only links the files, which plays in JupyterLab started from the repository root.

In [2]:
CACHE = Path("dataset/cache/98f5fa6e66")
SPLITS = "configs/splits/default.yaml"
SEED = 20261109  # lift selection
RULES = {  # one lift per rule, each from a different subject (test subjects only)
    "typical": {"rt_percentile": [45, 55]},
    "slow reaction": {"rt_percentile": [95, 100]},
    "P7": {"subject": "P7"},
    "heavy weight": {"weight_g": 660},
}
CHANNELS = ["Cz", "C3", "Fp1", "O1"]  # uncleaned broadband CAR
CLEANED_CHANNELS = ["Fp1"]  # the same channel after ICA ocular cleaning
PREDICTION_RUNS = {  # label -> LOSO run, seed 1
    "template_cue": "output/diagnostics/baselines/template_cue/20261104-234509",
    "CNN-LSTM direct": "output/diagnostics/history/cnn_lstm/history_200/20261105-210722_seed1",
    "CNN-LSTM residual": "output/diagnostics/pre-design/residual/cnn_lstm/20261106-155151_seed1",
    "EEGNet residual": "output/diagnostics/pre-design/residual/eegnet/20261106-160658_seed1",
    "CNN-LSTM residual, ICA": "output/diagnostics/final-checks/ocular/cnn_lstm_residual/20261108-110400_seed1",
}
FPS = 25
SPEED = 1.0  # 1 = real time
PRE_S, POST_S = 1.0, 0.5  # from LED on - PRE_S to hand stop + POST_S
TOPOMAP = True
EMBED_VIDEO = True
OUT = Path("output/diagnostics/visualization/timelapse") / make_run_id()

## Prediction sources

| label | model | input | output |
|---|---|---|---|
| template_cue | no trained model: mean training-subject trajectory vs time since LED on | LED-on time only | wrist position |
| CNN-LSTM direct | CNN-LSTM (temporal conv, spatial conv, LSTM, linear), D11 budget | 2 s EEG, 32 channels, CAR, 0.1-40 Hz, uncleaned | wrist position |
| CNN-LSTM residual | same network | same input | target minus template_cue, added back to the template |
| EEGNet residual | EEGNet | same input | same as CNN-LSTM residual |
| CNN-LSTM residual, ICA | same as CNN-LSTM residual | ICA ocular-cleaned EEG | same |

The cell below prints the LOSO test r of each run (seed 1, mean over test subjects). Seed-averaged scores are in `docs/diagnostics-results.md`.

In [3]:
for label, run in PREDICTION_RUNS.items():
    assert (Path(run) / "predictions.npz").exists(), f"{label}: no predictions.npz in {run}"
    cfg = json.loads((Path(run) / "metrics.json").read_text(encoding="utf-8"))
    print(f"{label:24s} {cfg['baseline']:15s} LOSO r {cfg['aggregate']['r']['mean']:.3f}  {run}")

template_cue             template_cue    LOSO r 0.803  output/diagnostics/baselines/template_cue/20261104-234509
CNN-LSTM direct          cnn_lstm        LOSO r 0.601  output/diagnostics/history/cnn_lstm/history_200/20261105-210722_seed1
CNN-LSTM residual        cnn_lstm        LOSO r 0.812  output/diagnostics/pre-design/residual/cnn_lstm/20261106-155151_seed1
EEGNet residual          eegnet          LOSO r 0.814  output/diagnostics/pre-design/residual/eegnet/20261106-160658_seed1
CNN-LSTM residual, ICA   cnn_lstm        LOSO r 0.796  output/diagnostics/final-checks/ocular/cnn_lstm_residual/20261108-110400_seed1


## Lift selection

Lifts flagged `onset_mismatch` or `filter_transient_broadband` are left out. RT percentiles are over all eligible lifts of the test subjects.

In [4]:
test_subjects = main_subjects(load_split_config(SPLITS))
trials = {s: load_subject(CACHE, s, arrays=[])[1] for s in test_subjects}
selection = select_lifts(trials, RULES, SEED)
OUT.mkdir(parents=True, exist_ok=False)
selection.to_csv(OUT / "selection.csv", index=False)
selection

,rule,subject,row,series,lift,weight_g,surface,rt_s
0,typical,P8,23,1,24,165,sandpaper,0.394
1,slow reaction,P2,42,2,15,330,suede,1.122
2,P7,P7,222,7,31,330,sandpaper,0.716
3,heavy weight,P11,113,4,18,660,sandpaper,0.406


## Rendering

Each video is rendered off screen to MP4 (H.264, `FPS` frames per second). The cursor advances `SPEED / FPS` seconds per frame, which is 4 samples of the 100 Hz data at real time.

In [5]:
videos = []
for sel in selection.itertuples():
    bundle = load_lift(CACHE, sel.subject, sel.row, CHANNELS, CLEANED_CHANNELS, PREDICTION_RUNS)
    name = f"{sel.rule.replace(' ', '_')}_{sel.subject}_S{sel.series}_lift{sel.lift}.mp4"
    info = animate_lift(bundle, OUT / name, fps=FPS, speed=SPEED, pre_s=PRE_S, post_s=POST_S, topomap=TOPOMAP)
    videos.append({"rule": sel.rule, "bundle": bundle, **info})
    print(f"{name}: {info['n_frames']} frames, {info['t_start_s']:.2f} to {info['t_end_s']:.2f} s after LED on")

D:\research\university\Brain Signal to 3D Trajectory Representation\physio-ssl-trajectory\src\eegtraj\visualization\timelapse.py:342: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  info.set_montage("standard_1005")


typical_P8_S1_lift24.mp4: 180 frames, -1.00 to 6.16 s after LED on


D:\research\university\Brain Signal to 3D Trajectory Representation\physio-ssl-trajectory\src\eegtraj\visualization\timelapse.py:342: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  info.set_montage("standard_1005")


slow_reaction_P2_S2_lift15.mp4: 257 frames, -1.00 to 9.24 s after LED on


D:\research\university\Brain Signal to 3D Trajectory Representation\physio-ssl-trajectory\src\eegtraj\visualization\timelapse.py:342: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  info.set_montage("standard_1005")


P7_P7_S7_lift31.mp4: 197 frames, -1.00 to 6.84 s after LED on


D:\research\university\Brain Signal to 3D Trajectory Representation\physio-ssl-trajectory\src\eegtraj\visualization\timelapse.py:342: FutureWarning: Montage name 'standard_1005' is deprecated and will be removed in MNE 1.14. Use 'colin27_1005' instead.
  info.set_montage("standard_1005")


heavy_weight_P11_S4_lift18.mp4: 213 frames, -1.00 to 7.48 s after LED on


## Timing check

Checks per video: the file holds the expected number of frames and duration; the cursor sample of every frame lies on the 100 Hz grid at the frame time; the frame nearest each event shows the cursor within half a frame of the event time.

In [6]:
import imageio_ffmpeg
import numpy as np

rows = []
for v in videos:
    n_file, secs = imageio_ffmpeg.count_frames_and_secs(v["path"])
    t = v["bundle"].t_s
    cursor_err = np.abs(t[v["frame_samples"]] - v["frame_times_s"]).max()
    event_err = max(abs(v["frame_times_s"][k] - v["bundle"].events[e]) for e, k in v["event_frames"].items())
    rows.append({"video": Path(v["path"]).name, "frames": v["n_frames"], "frames_in_file": n_file,
                 "seconds_in_file": round(secs, 2), "expected_s": round(v["n_frames"] / FPS, 2),
                 "max_cursor_vs_sample_s": round(float(cursor_err), 4), "max_event_vs_frame_s": round(float(event_err), 4),
                 "samples_per_frame": sorted(set(np.diff(v["frame_samples"]).tolist()))})
    assert n_file == v["n_frames"]
    assert cursor_err < 0.5 / 100 and event_err <= 0.5 * SPEED / FPS + 1e-9
timing = pd.DataFrame(rows)
timing.to_csv(OUT / "timing.csv", index=False)
timing

,video,frames,frames_in_file,seconds_in_file,expected_s,max_cursor_vs_sample_s,max_event_vs_frame_s,samples_per_frame
0,typical_P8_S1_lift24.mp4,180,180,7.20,7.20,0.0,0.018,[4]
1,slow_reaction_P2_S2_lift15.mp4,257,257,10.28,10.28,0.0,0.016,[4]
2,P7_P7_S7_lift31.mp4,197,197,7.88,7.88,0.0,0.018,[4]
3,heavy_weight_P11_S4_lift18.mp4,213,213,8.52,8.52,0.0,0.018,[4]


## Videos

In [7]:
for v in videos:
    b = v["bundle"]
    display(Markdown(f"**{v['rule']}**: {b.subject} S{b.series} lift {b.lift}, {b.weight_g} g, {b.surface}, "
                     f"RT {b.rt_s:.2f} s"))
    display(Video(v["path"], embed=EMBED_VIDEO, width=960, html_attributes="controls loop"))

**typical**: P8 S1 lift 24, 165 g, sandpaper, RT 0.39 s

**slow reaction**: P2 S2 lift 15, 330 g, suede, RT 1.12 s

**P7**: P7 S7 lift 31, 330 g, sandpaper, RT 0.72 s

**heavy weight**: P11 S4 lift 18, 660 g, sandpaper, RT 0.41 s

## Output size

In [8]:
sizes = {p.name: p.stat().st_size / 1e6 for p in sorted(OUT.glob("*.mp4"))}
for k, v in sizes.items():
    print(f"{k}: {v:.2f} MB")
print(f"total {sum(sizes.values()):.2f} MB in {OUT}")

heavy_weight_P11_S4_lift18.mp4: 0.43 MB
P7_P7_S7_lift31.mp4: 0.40 MB
slow_reaction_P2_S2_lift15.mp4: 0.59 MB
typical_P8_S1_lift24.mp4: 0.40 MB
total 1.81 MB in output\diagnostics\visualization\timelapse\20261109-105955
